# 🌳 Lab W5-3 — Entropy, Information Gain และต้นไม้ที่อธิบายได้

**รายวิชาระบบสนับสนุนการตัดสินใจ · สัปดาห์ที่ 5 — Data Mining I**

Lab นี้ใช้คู่กับสื่อจำลอง **Decision Tree Grower** (`/sims/tree-grower`)
ตัวเลขที่คุณคำนวณได้ในสมุดเล่มนี้ต้องตรงกับตัวเลขบนหน้าจอสื่อจำลองทุกหลัก

## สิ่งที่จะได้เรียนรู้
1. คำนวณ **entropy** และ **information gain** ด้วยมือ ไม่ใช่เรียกไลบรารี
2. อธิบายว่าเหตุใดต้นไม้จึงเลือกตัวแปรที่มันเลือก
3. แสดงให้เห็นว่า **ความแม่นสูงไม่ได้แปลว่าโมเดลมีประโยชน์**
4. หา **จุดที่ต้นไม้เริ่มจดจำเสียงรบกวน** (overfitting) ด้วยตัวเลข

## ข้อมูล
`churn.csv` — ลูกค้าโทรคมนาคม 3,000 ราย พร้อมผลว่าเลิกใช้บริการหรือไม่

In [ ]:
import math

import numpy as np
import pandas as pd

pd.set_option("display.float_format", lambda v: f"{v:,.4f}")

URL = ("https://raw.githubusercontent.com/babankbro/ksu-dss-course/"
       "master/datasets/week05/churn.csv")
df = pd.read_csv(URL)

print(f"จำนวนลูกค้า     : {len(df):,}")
print(f"เลิกใช้บริการ   : {int(df.churned.sum()):,} ({df.churned.mean()*100:.2f}%)")
print(f"\nประเภทสัญญา:\n{df.contract_type.value_counts().to_string()}")
df.head(5)

## ส่วนที่ 1 — Entropy คืออะไรกันแน่

entropy วัด **ความไม่แน่นอน** ของกลุ่ม — สูงสุดเมื่อผสมกันครึ่งต่อครึ่ง
และเป็นศูนย์เมื่อทุกคนในกลุ่มเหมือนกันหมด

### 🧑‍💻 งานที่ 1
เขียนฟังก์ชัน `entropy(pos, n)` ที่รับจำนวนบวกและจำนวนทั้งหมด
แล้วคืนค่า entropy แบบฐานสอง (ต้องจัดการกรณี p = 0 และ p = 1 ให้คืน 0)

ทดสอบว่า `entropy(50, 100) == 1.0` และ `entropy(0, 100) == 0.0`
แล้วคำนวณ entropy ของข้อมูลทั้งชุด

*เฉลยที่ถูกต้อง: entropy ที่ราก = 0.9083*

In [ ]:
def entropy(pos: int, n: int) -> float:
    if n == 0:
        return 0.0
    p = pos / n
    if p in (0.0, 1.0):
        return 0.0
    return -p * math.log2(p) - (1 - p) * math.log2(1 - p)


assert entropy(50, 100) == 1.0
assert entropy(0, 100) == 0.0
assert entropy(100, 100) == 0.0
print("✓ ฟังก์ชัน entropy ผ่านการทดสอบ")

root_pos, root_n = int(df.churned.sum()), len(df)
root_ent = entropy(root_pos, root_n)
print(f"\nที่ราก: {root_pos:,} จาก {root_n:,} ราย ({root_pos/root_n*100:.2f}%)")
print(f"entropy = {root_ent:.4f}")
print(f"→ ใกล้ 1.0 แปลว่ากลุ่มนี้ยังปนกันมาก ทายอะไรก็ยังผิดบ่อย")

## ส่วนที่ 2 — Information Gain

### 🧑‍💻 งานที่ 2
เขียนฟังก์ชัน `info_gain(data, mask)` ที่คืนค่า
information gain ของการแบ่งสองทางตามเงื่อนไข `mask`

สูตร: `IG = entropy(พ่อ) − [ (n_ซ้าย/n) × entropy(ซ้าย) + (n_ขวา/n) × entropy(ขวา) ]`

แล้วคำนวณ IG ของเงื่อนไข **ประเภทสัญญา = รายเดือน**

*เฉลยที่ถูกต้อง: IG = 0.1763 · ซ้าย 1,577 ราย (53.39%) · ขวา 1,423 ราย (9.07%)*

In [ ]:
def node(data):
    """สรุปโหนดหนึ่งโหนด"""
    pos, n = int(data.churned.sum()), len(data)
    return {"n": n, "pos": pos, "rate": pos / n if n else 0.0, "ent": entropy(pos, n)}


def info_gain(data, mask):
    n = len(data)
    left, right = data[mask], data[~mask]
    if len(left) == 0 or len(right) == 0:
        return None
    l, r = node(left), node(right)
    weighted = l["n"] / n * l["ent"] + r["n"] / n * r["ent"]
    return {
        "gain": entropy(int(data.churned.sum()), n) - weighted,
        "n_ซ้าย": l["n"], "อัตราซ้าย %": l["rate"] * 100, "ent ซ้าย": l["ent"],
        "n_ขวา": r["n"], "อัตราขวา %": r["rate"] * 100, "ent ขวา": r["ent"],
    }


g = info_gain(df, df.contract_type == "รายเดือน")
for k, v in g.items():
    print(f"  {k:<14} {v:>12,.4f}")

## ส่วนที่ 3 — จัดอันดับผู้สมัครทุกตัว

### 🧑‍💻 งานที่ 3
ประเมิน information gain ของเงื่อนไขทั้ง 11 ข้อด้านล่าง แล้วเรียงจากมากไปน้อย

ก่อนรันโค้ด **ให้เดาก่อน** ว่าเงื่อนไขใดจะมาเป็นอันดับหนึ่ง แล้วจดคำตอบไว้

*เฉลยที่ถูกต้อง: อันดับ 1 คือ ประเภทสัญญา = รายเดือน (0.1763)
อันดับสุดท้ายคือ ใบแจ้งหนี้อิเล็กทรอนิกส์ (0.0003)*

In [ ]:
CANDIDATES = {
    "ประเภทสัญญา = รายเดือน": lambda d: d.contract_type == "รายเดือน",
    "ประเภทสัญญา = 2 ปี": lambda d: d.contract_type == "2 ปี",
    "อายุการใช้งาน ≤ 12 เดือน": lambda d: d.tenure_months <= 12,
    "อายุการใช้งาน ≤ 24 เดือน": lambda d: d.tenure_months <= 24,
    "อายุการใช้งาน ≤ 36 เดือน": lambda d: d.tenure_months <= 36,
    "ค่าบริการ > 1,000 บาท": lambda d: d.monthly_charge > 1000,
    "แจ้งปัญหา ≥ 2 ครั้ง": lambda d: d.support_tickets >= 2,
    "แจ้งปัญหา ≥ 3 ครั้ง": lambda d: d.support_tickets >= 3,
    "อินเทอร์เน็ต = ไฟเบอร์": lambda d: d.internet_type == "ไฟเบอร์",
    "ไม่ตัดบัญชีอัตโนมัติ": lambda d: d.auto_payment == "ไม่",
    "ใบแจ้งหนี้อิเล็กทรอนิกส์": lambda d: d.paperless_billing == "ใช่",
}

In [ ]:
def rank_splits(data):
    rows = []
    for name, fn in CANDIDATES.items():
        r = info_gain(data, fn(data))
        if r:
            rows.append({"เงื่อนไข": name, **r})
    return (pd.DataFrame(rows).set_index("เงื่อนไข")
            .sort_values("gain", ascending=False))


ranked = rank_splits(df)
print(ranked[["gain", "n_ซ้าย", "อัตราซ้าย %", "n_ขวา", "อัตราขวา %"]].to_string())

print(f"""
สิ่งที่ต้องสังเกต
-----------------
อันดับ 1 ({ranked.index[0]}) ได้ gain {ranked.gain.iloc[0]:.4f}
ซึ่งมากกว่าอันดับ 2 ถึง {ranked.gain.iloc[0]/ranked.gain.iloc[1]:.1f} เท่า

อันดับสุดท้าย ({ranked.index[-1]}) ได้ gain {ranked.gain.iloc[-1]:.4f}
ซึ่งแทบเป็นศูนย์ — ตัวแปรนี้ไม่ได้บอกอะไรเลยเกี่ยวกับการเลิกใช้บริการ

นักศึกษาส่วนใหญ่เดาว่า 'ค่าบริการสูง' หรือ 'แจ้งปัญหาบ่อย' จะมาอันดับหนึ่ง
เพราะฟังดูเป็นเหตุเป็นผล แต่ทั้งสองได้ gain เพียง {ranked.loc['ค่าบริการ > 1,000 บาท','gain']:.4f}
และ {ranked.loc['แจ้งปัญหา ≥ 2 ครั้ง','gain']:.4f} ตามลำดับ

→ ความรู้สึกว่าตัวแปรใดสำคัญ ไม่ใช่หลักฐาน · information gain คือหลักฐาน
""")

## ส่วนที่ 4 — ปลูกต้นไม้สองชั้น

### 🧑‍💻 งานที่ 4
แบ่งด้วยเงื่อนไขที่ดีที่สุดที่ราก แล้วหาเงื่อนไขที่ดีที่สุดของแต่ละกิ่ง
จากนั้นสร้างตารางแสดงใบทั้ง 4 ใบ พร้อมอัตราเลิกใช้บริการของแต่ละใบ

*เฉลยที่ถูกต้อง: กิ่ง "รายเดือน" แบ่งต่อด้วยอายุการใช้งาน ≤ 24 เดือน (gain 0.0549)
ได้ใบที่มีอัตราเลิกใช้ 72.78%*

In [ ]:
best_root = ranked.index[0]
root_fn = CANDIDATES[best_root]

leaves = []
for side, mask in [("ใช่", root_fn(df)), ("ไม่ใช่", ~root_fn(df))]:
    branch = df[mask]
    br = rank_splits(branch)
    best2 = br.index[0]
    print(f"กิ่ง “{best_root} = {side}”  (n={len(branch):,}, "
          f"อัตรา {branch.churned.mean()*100:.2f}%, entropy {node(branch)['ent']:.4f})")
    print(f"  → เงื่อนไขที่ดีที่สุด: {best2}  (gain {br.gain.iloc[0]:.4f})\n")

    fn2 = CANDIDATES[best2]
    for side2, m2 in [("ใช่", fn2(branch)), ("ไม่ใช่", ~fn2(branch))]:
        leaf = branch[m2]
        leaves.append({
            "ใบ": f"{side} → {best2} {side2}",
            "จำนวน": len(leaf),
            "อัตราเลิกใช้ %": leaf.churned.mean() * 100,
            "ทำนายว่า": "เลิกใช้" if leaf.churned.mean() > 0.5 else "อยู่ต่อ",
            "ทายถูก": int(max(leaf.churned.sum(), len(leaf) - leaf.churned.sum())),
        })

tree = pd.DataFrame(leaves).set_index("ใบ")
print(tree.to_string())

tree_acc = tree["ทายถูก"].sum() / len(df)
majority_acc = max(root_pos, root_n - root_pos) / root_n
print(f"\nความแม่นของต้นไม้        : {tree_acc*100:.2f}%")
print(f"ทายเสียงข้างมากอย่างเดียว : {majority_acc*100:.2f}%")
print(f"ดีขึ้น                    : {(tree_acc-majority_acc)*100:.2f} จุด")

> **อย่าเพิ่งดีใจกับความแม่น** การทายว่า "อยู่ต่อ" ทุกรายได้ 67.63% ทันทีโดยไม่ต้องมีโมเดล
> ต้นไม้ที่เพิ่งปลูกดีขึ้นเพียงไม่กี่จุด
>
> **คุณค่าจริงของต้นไม้ต้นนี้ไม่ได้อยู่ที่ความแม่น** แต่อยู่ที่มันชี้กลุ่มลูกค้า
> ที่มีอัตราเลิกใช้ 72.78% ให้ทีมการตลาดไปทำงานต่อได้ —
> กลุ่มที่มีขนาดพอจะทำแคมเปญ และมีเหตุผลที่อธิบายให้ผู้บริหารฟังได้ในประโยคเดียว

## ส่วนที่ 5 — ความแม่นที่ไม่มีประโยชน์

### 🧑‍💻 งานที่ 5
คำนวณสำหรับใบที่มีอัตราเลิกใช้สูงที่สุด

1. ถ้าทำแคมเปญรักษาลูกค้าเฉพาะใบนี้ จะครอบคลุมลูกค้าที่จะเลิกใช้ทั้งหมดกี่เปอร์เซ็นต์ (recall)
2. ถ้าแคมเปญมีต้นทุน 200 บาท/ราย และรักษาลูกค้าไว้ได้ 1 ราย มีมูลค่า 3,000 บาท
   การทำแคมเปญกับใบนี้คุ้มหรือไม่ (สมมติแคมเปญได้ผล 30% ของผู้ที่จะเลิกใช้)
3. เทียบกับการทำแคมเปญกับลูกค้าทุกราย — แบบใดคุ้มกว่า
   **คำเตือน: คำตอบอาจไม่ใช่อย่างที่คุณคิด** ให้ตอบตามตัวเลขที่ได้จริง
4. ถ้างบประมาณถูกจำกัดให้ติดต่อลูกค้าได้เท่ากับขนาดของใบที่เสี่ยงที่สุดเท่านั้น
   คำตอบข้อ 3 เปลี่ยนไปหรือไม่ และส่วนต่างเป็นเท่าไร

In [ ]:
COST_PER = 200
VALUE_SAVED = 3000
EFFECT = 0.30

top = tree.sort_values("อัตราเลิกใช้ %", ascending=False).iloc[0]
n_leaf = int(top["จำนวน"])
churners_in_leaf = round(n_leaf * top["อัตราเลิกใช้ %"] / 100)
total_churners = root_pos

print(f"ใบที่เสี่ยงที่สุด : {top.name}")
print(f"  ลูกค้าในใบ            : {n_leaf:,} ราย")
print(f"  ในนั้นจะเลิกใช้จริง    : {churners_in_leaf:,} ราย")
print(f"  recall ของแคมเปญ      : {churners_in_leaf/total_churners*100:.2f}% "
      f"ของผู้ที่จะเลิกใช้ทั้งหมด {total_churners:,} ราย\n")


def campaign(n_target, n_churners, label):
    cost = n_target * COST_PER
    saved = n_churners * EFFECT
    benefit = saved * VALUE_SAVED
    print(f"{label}")
    print(f"  ต้นทุน  : {cost:>12,.0f} บาท ({n_target:,} ราย)")
    print(f"  รักษาได้ : {saved:>12,.1f} ราย")
    print(f"  มูลค่า   : {benefit:>12,.0f} บาท")
    print(f"  กำไรสุทธิ: {benefit-cost:>12,.0f} บาท  "
          f"(ROI {(benefit-cost)/cost*100:+.1f}%)\n")
    return benefit - cost


p1 = campaign(n_leaf, churners_in_leaf, "ทำแคมเปญเฉพาะใบที่เสี่ยงที่สุด")
p2 = campaign(root_n, total_churners, "ทำแคมเปญกับลูกค้าทุกราย")

print(f"ต่างกัน {p1-p2:+,.0f} บาท → "
      f"{'เจาะเฉพาะกลุ่มเสี่ยงได้กำไรรวมมากกว่า' if p1 > p2 else 'หว่านทั้งหมดได้กำไรรวมมากกว่า'}")

print(f"""
บทเรียน — คำตอบขึ้นกับว่าคุณถามคำถามอะไร
==========================================
ถ้าถามว่า 'กำไรรวมสูงสุด'  → หว่านทั้งหมดชนะ ({p2:,.0f} เทียบ {p1:,.0f} บาท)
ถ้าถามว่า 'ใช้เงินคุ้มที่สุด' → เจาะกลุ่มเสี่ยงชนะขาด (ROI {(p1)/(n_leaf*COST_PER)*100:.0f}%
                              เทียบ {(p2)/(root_n*COST_PER)*100:.0f}%)

นักศึกษามักคาดว่าการเจาะกลุ่มเสี่ยงต้องชนะเสมอ ซึ่งไม่จริง —
เมื่อแคมเปญมี ROI เป็นบวกกับ 'ทุกกลุ่ม' การหว่านให้กว้างที่สุดย่อมได้กำไรรวมมากกว่า
โมเดลจะสร้างมูลค่าก็ต่อเมื่อมี 'ข้อจำกัด' อยู่จริง

ทดสอบข้อนี้ด้วยตัวเลข: ถ้างบประมาณจำกัดที่ {n_leaf*COST_PER:,} บาท
  เจาะกลุ่มเสี่ยง : ติดต่อได้ {n_leaf:,} ราย → กำไร {p1:,.0f} บาท
  หว่านแบบสุ่ม    : ติดต่อได้ {n_leaf:,} ราย เช่นกัน แต่เจอผู้ที่จะเลิกใช้เพียง
                    {n_leaf*root_pos/root_n:.0f} ราย → กำไร
                    {n_leaf*root_pos/root_n*EFFECT*VALUE_SAVED - n_leaf*COST_PER:,.0f} บาท

→ ภายใต้งบเท่ากัน โมเดลสร้างมูลค่าเพิ่ม
  {p1 - (n_leaf*root_pos/root_n*EFFECT*VALUE_SAVED - n_leaf*COST_PER):,.0f} บาท

ตัววัดที่ควรรายงานให้ผู้บริหารจึงไม่ใช่ accuracy แต่คือ
กำไรสุทธิภายใต้งบที่มีจริง · จำนวนลูกค้าที่ต้องติดต่อ · และสัดส่วนที่ครอบคลุมได้
""")

## ส่วนที่ 6 — ต้นไม้ลึกแค่ไหนถึงจะพอ

### 🧑‍💻 งานที่ 6
ใช้ `DecisionTreeClassifier` ของ scikit-learn ปลูกต้นไม้ที่ความลึก 1 ถึง 20
แบ่งข้อมูล 70/30 แล้ววาดกราฟความแม่นของชุดฝึกกับชุดทดสอบเทียบกัน

หาจุดที่ทั้งสองเส้นเริ่มแยกจากกัน แล้วอธิบายว่าเกิดอะไรขึ้นที่จุดนั้น

In [ ]:
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, export_text

X = pd.get_dummies(
    df[["contract_type", "tenure_months", "monthly_charge", "support_tickets",
        "internet_type", "paperless_billing", "auto_payment"]],
    drop_first=True,
)
y = df.churned
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

rows = []
for depth in range(1, 21):
    clf = DecisionTreeClassifier(criterion="entropy", max_depth=depth, random_state=42)
    clf.fit(Xtr, ytr)
    rows.append({
        "ความลึก": depth,
        "ชุดฝึก %": accuracy_score(ytr, clf.predict(Xtr)) * 100,
        "ชุดทดสอบ %": accuracy_score(yte, clf.predict(Xte)) * 100,
        "จำนวนใบ": clf.get_n_leaves(),
    })

curve = pd.DataFrame(rows).set_index("ความลึก")
curve["ช่องว่าง"] = curve["ชุดฝึก %"] - curve["ชุดทดสอบ %"]
print(curve.to_string())

best_depth = curve["ชุดทดสอบ %"].idxmax()
print(f"\nความลึกที่ดีที่สุดบนชุดทดสอบ : {best_depth} "
      f"({curve.loc[best_depth,'ชุดทดสอบ %']:.2f}%)")
print(f"ที่ความลึก 20: ชุดฝึก {curve.loc[20,'ชุดฝึก %']:.2f}% "
      f"แต่ชุดทดสอบ {curve.loc[20,'ชุดทดสอบ %']:.2f}% "
      f"(ห่างกัน {curve.loc[20,'ช่องว่าง']:.2f} จุด · {int(curve.loc[20,'จำนวนใบ']):,} ใบ)")

print("\nต้นไม้ที่ความลึก 2 — อ่านออกได้ด้วยตาเปล่า")
clf2 = DecisionTreeClassifier(criterion="entropy", max_depth=2, random_state=42)
clf2.fit(Xtr, ytr)
print(export_text(clf2, feature_names=list(X.columns), decimals=3))

print("""
เกิดอะไรขึ้นเมื่อต้นไม้ลึกขึ้น
-------------------------------
ความแม่นบนชุดฝึกไต่ขึ้นเรื่อย ๆ จนเกือบ 100% เพราะต้นไม้แตกใบได้จนเหลือ
ไม่กี่คนต่อใบ และสุดท้ายก็ 'จำ' ลูกค้าแต่ละคนได้

แต่ความแม่นบนชุดทดสอบกลับลดลง เพราะสิ่งที่มันจำคือ 'เสียงรบกวน' ของชุดฝึก
ไม่ใช่รูปแบบที่มีอยู่จริงในประชากร

ช่องว่างระหว่างสองเส้นคือมาตรวัดของ overfitting ที่ตรงไปตรงมาที่สุด
และเป็นเหตุผลที่ต้องจำกัดความลึก หรือใช้ ensemble อย่าง Random Forest
ซึ่งจะได้เรียนในสัปดาห์ที่ 6
""")

## ส่วนที่ 7 — แปลต้นไม้เป็นภาษาคน

### 🧑‍💻 งานที่ 7 (เขียนเป็นข้อความ)

1. เขียนกฎที่ได้จากต้นไม้ความลึก 2 เป็น **ภาษาไทยธรรมดา** ไม่เกิน 4 บรรทัด
   ที่ทีมการตลาดอ่านแล้วลงมือทำได้ทันที
2. ต้นไม้บอกว่าลูกค้าสัญญารายเดือนที่ใช้บริการไม่ถึง 2 ปีเสี่ยงสูงมาก
   นี่เป็น **ความสัมพันธ์** หรือ **เหตุและผล** — และการแยกสองอย่างนี้สำคัญอย่างไร
   ต่อการออกแบบแคมเปญ
3. ถ้าบริษัทยกเลิกสัญญารายเดือนทั้งหมดตามคำแนะนำของต้นไม้ จะเกิดอะไรขึ้น
   และเหตุใดต้นไม้จึงไม่สามารถเตือนเรื่องนี้ได้

In [ ]:
print("""ตัวอย่างคำตอบข้อ 2 และ 3
==========================

ข้อ 2 — ความสัมพันธ์ ไม่ใช่เหตุและผล
--------------------------------------
ต้นไม้เห็นเพียงว่า 'สัญญารายเดือน' กับ 'การเลิกใช้บริการ' เกิดร่วมกันบ่อย
มันไม่มีทางรู้ว่าอะไรทำให้อะไรเกิด

คำอธิบายที่เป็นไปได้อย่างน้อย 3 แบบ ซึ่งข้อมูลชุดนี้แยกไม่ออก
  ก. สัญญารายเดือนทำให้เลิกง่าย (เหตุ → ผล จริง)
  ข. คนที่ตั้งใจจะอยู่สั้นอยู่แล้ว จึงเลือกสัญญารายเดือนตั้งแต่ต้น (ผลย้อนกลับ)
  ค. ทั้งสองอย่างเกิดจากตัวแปรที่สาม เช่น ความไม่แน่นอนของรายได้

ผลต่อการออกแบบแคมเปญต่างกันสิ้นเชิง
  ถ้าเป็น ก. → เสนอส่วนลดให้ย้ายไปสัญญาระยะยาว จะได้ผล
  ถ้าเป็น ข. → เสนอไปก็ไม่มีใครรับ เพราะเขารู้ตัวว่าจะอยู่สั้น
                เงินที่ใช้จะสูญเปล่าทั้งหมด

วิธีเดียวที่แยกได้คือ 'ทดลอง' — สุ่มเลือกลูกค้ากลุ่มเสี่ยงครึ่งหนึ่งให้ข้อเสนอ
อีกครึ่งไม่ให้ แล้ววัดส่วนต่าง นี่คือเหตุผลที่ A/B test ยังจำเป็น
แม้จะมีโมเดลที่แม่นแล้วก็ตาม

ข้อ 3 — เหตุใดต้นไม้จึงเตือนไม่ได้
------------------------------------
ถ้ายกเลิกสัญญารายเดือนทั้งหมด ลูกค้ากลุ่มนั้นจะไม่ได้ย้ายไปสัญญา 2 ปี
แต่จะย้ายไปหาคู่แข่งที่ยังมีสัญญารายเดือนให้ — บริษัทจะเสียลูกค้าเร็วกว่าเดิม

ต้นไม้เตือนไม่ได้เพราะมันเรียนรู้จากข้อมูลของโลกที่ 'ยังมีสัญญารายเดือนอยู่'
เมื่อเปลี่ยนนโยบาย โลกที่โมเดลเคยเห็นก็หายไป และความสัมพันธ์ทั้งหมดที่มันจับได้
ก็ไม่รับประกันว่าจะยังจริง

นี่คือข้อจำกัดพื้นฐานของโมเดลทำนายทุกตัว — มันตอบได้ว่า
'ถ้าโลกยังเป็นแบบเดิม จะเกิดอะไรขึ้น' แต่ตอบไม่ได้ว่า
'ถ้าเราเปลี่ยนโลก จะเกิดอะไรขึ้น' คำถามหลังต้องใช้การทดลองหรือแบบจำลองเชิงสาเหตุ
""")

---
## ✅ เกณฑ์การส่งงาน

| องค์ประกอบ | คะแนน |
|---|:--:|
| งานที่ 1 — ฟังก์ชัน entropy ที่ผ่านการทดสอบ | 2 |
| งานที่ 2 — ฟังก์ชัน information gain ถูกต้อง | 3 |
| งานที่ 3 — จัดอันดับครบ 11 เงื่อนไขและเดาก่อนรัน | 3 |
| งานที่ 4 — ปลูกต้นไม้สองชั้นและสรุปใบทั้ง 4 | 3 |
| งานที่ 5 — วิเคราะห์ความคุ้มค่าของแคมเปญ | 3 |
| งานที่ 6 — กราฟความลึกกับ overfitting และคำอธิบาย | 3 |
| งานที่ 7 — แปลเป็นภาษาคน และแยกความสัมพันธ์ออกจากเหตุผล | 3 |
| **รวม** | **20** |

> 💡 ตัวเลขทุกตัวในสมุดเล่มนี้ต้องตรงกับที่แสดงบนสื่อจำลอง `/sims/tree-grower`
> ถ้าไม่ตรง แปลว่ามีขั้นตอนใดขั้นตอนหนึ่งผิด — ให้ย้อนกลับไปตรวจก่อนส่ง